### Notebook Scope

This notebook focuses on **data modeling and KPI view creation**.  
Exploratory analysis and data validation are covered in a separate notebook.


In [1]:
import sqlite3

conn = sqlite3.connect("../data/travel.sqlite")

cursor = conn.cursor()


In [2]:
# Create base analytical view

cursor.executescript("""
DROP VIEW IF EXISTS analytics_flights_base;

CREATE VIEW analytics_flights_base AS
SELECT
    flight_id,
    flight_no,
    departure_airport,
    arrival_airport,
    aircraft_code,
    status,

    -- scheduled timestamps (normalized)
    SUBSTR(scheduled_departure, 1, 19) AS scheduled_departure_ts,
    SUBSTR(scheduled_arrival, 1, 19)   AS scheduled_arrival_ts,

    -- actual timestamps (normalized, NULL-safe)
    CASE WHEN actual_departure != '\\N'
         THEN SUBSTR(actual_departure, 1, 19)
         ELSE NULL END AS actual_departure_ts,

    CASE WHEN actual_arrival != '\\N'
         THEN SUBSTR(actual_arrival, 1, 19)
         ELSE SUBSTR(NULL, 1, 19) END AS actual_arrival_ts,

    -- departure delay in minutes
    CASE
        WHEN status != 'Cancelled' AND actual_departure != '\\N'
        THEN (JULIANDAY(SUBSTR(actual_departure, 1, 19))
            - JULIANDAY(SUBSTR(scheduled_departure, 1, 19))) * 1440
        ELSE NULL
    END AS dep_delay_min,

    -- arrival delay in minutes
    CASE
        WHEN status != 'Cancelled' AND actual_arrival != '\\N'
        THEN (JULIANDAY(SUBSTR(actual_arrival, 1, 19))
            - JULIANDAY(SUBSTR(scheduled_arrival, 1, 19))) * 1440
        ELSE NULL
    END AS arr_delay_min

FROM flights;
""")

conn.commit()

In [3]:
cursor.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(dep_delay_min) AS flights_with_dep_delay,
    COUNT(arr_delay_min) AS flights_with_arr_delay
FROM analytics_flights_base;
""")

print(cursor.fetchone())

(33121, 16765, 16707)


### Base Analytical View — `analytics_flights_base`

**Purpose**  
Provides a clean, reusable foundation for all analysis and dashboards.

**What it does**
- Preserves one row per flight
- Normalizes timestamps and missing values (`'\N'`)
- Handles SQLite timezone limitations
- Computes departure and arrival delays once

**Why it matters**
- Prevents duplicated logic
- Ensures consistent KPIs
- Acts as the single source of truth for Excel and Tableau

**Validation**
- Total flights: **33,121**
- Flights with departure delay: **16,765**
- Flights with arrival delay: **16,707**


In [4]:
# Airport-level delay KPIs

cursor.executescript("""
DROP VIEW IF EXISTS airport_delay_kpis;

CREATE VIEW airport_delay_kpis AS
WITH dep AS (
    SELECT
        departure_airport AS airport_code,
        COUNT(dep_delay_min) AS dep_flights,
        ROUND(AVG(dep_delay_min), 2) AS avg_dep_delay_min
    FROM analytics_flights_base
    WHERE dep_delay_min IS NOT NULL
    GROUP BY departure_airport
),
arr AS (
    SELECT
        arrival_airport AS airport_code,
        COUNT(arr_delay_min) AS arr_flights,
        ROUND(AVG(arr_delay_min), 2) AS avg_arr_delay_min
    FROM analytics_flights_base
    WHERE arr_delay_min IS NOT NULL
    GROUP BY arrival_airport
)
SELECT
    COALESCE(dep.airport_code, arr.airport_code) AS airport_code,
    dep.dep_flights,
    dep.avg_dep_delay_min,
    arr.arr_flights,
    arr.avg_arr_delay_min
FROM dep
FULL OUTER JOIN arr
ON dep.airport_code = arr.airport_code;
""")

conn.commit()


In [5]:
cursor.execute("""
SELECT *
FROM airport_delay_kpis
ORDER BY avg_dep_delay_min DESC
LIMIT 10;
""")

for row in cursor.fetchall():
    print(row)


('ULY', 97, 24.4, 97, 18.49)
('BQS', 31, 22.81, 31, 30.26)
('LPK', 22, 22.68, 22, 11.95)
('CNN', 81, 19.25, 81, 8.14)
('UCT', 93, 18.71, 93, 7.33)
('NAL', 93, 17.81, 93, 5.3)
('SWT', 31, 16.68, 31, 16.29)
('CEE', 67, 16.54, 64, 13.33)
('CSY', 155, 16.52, 154, 9.94)
('KLF', 62, 16.31, 61, 9.72)



### Airport-Level Delay KPIs — `airport_delay_kpis`

**Business question**  
Which airports experience the highest average departure and arrival delays?

**How it works**
- Aggregates departure delays by departure airport
- Aggregates arrival delays by arrival airport
- Reports flight counts and average delays (minutes)

**Key observations**
- Smaller airports show higher average delays
- Departure delays are generally higher than arrival delays
- Limited delay recovery during flight

**Usage**
- Airport benchmarking
- Ranking dashboards in Tableau
- Identifying operational bottlenecks

In [6]:
# Time-based operational stress (by month)

cursor.executescript("""
DROP VIEW IF EXISTS time_stress_kpis_month;

CREATE VIEW time_stress_kpis_month AS
SELECT
    CAST(STRFTIME('%m', scheduled_departure_ts) AS INTEGER) AS month,
    COUNT(*) AS total_flights,
    COUNT(dep_delay_min) AS flights_with_dep_delay,
    ROUND(AVG(dep_delay_min), 2) AS avg_dep_delay_min,
    COUNT(arr_delay_min) AS flights_with_arr_delay,
    ROUND(AVG(arr_delay_min), 2) AS avg_arr_delay_min
FROM analytics_flights_base
GROUP BY month
ORDER BY month;
""")

conn.commit()


In [7]:
cursor.execute("""
SELECT *
FROM time_stress_kpis_month;
""")

for row in cursor.fetchall():
    print(row)

(7, 8690, 8686, 12.87, 8686, 12.86)
(8, 16835, 8079, 11.48, 8021, 11.49)
(9, 7596, 0, None, 0, None)


In [8]:
# Time-based operational stress (by weekday)

cursor.executescript("""
DROP VIEW IF EXISTS time_stress_kpis_weekday;

CREATE VIEW time_stress_kpis_weekday AS
SELECT
    CAST(STRFTIME('%w', scheduled_departure_ts) AS INTEGER) AS weekday,
    COUNT(*) AS total_flights,
    COUNT(dep_delay_min) AS flights_with_dep_delay,
    ROUND(AVG(dep_delay_min), 2) AS avg_dep_delay_min,
    COUNT(arr_delay_min) AS flights_with_arr_delay,
    ROUND(AVG(arr_delay_min), 2) AS avg_arr_delay_min
FROM analytics_flights_base
GROUP BY weekday
ORDER BY weekday;
""")

conn.commit()


In [9]:
cursor.execute("SELECT * FROM time_stress_kpis_weekday;")
for row in cursor.fetchall():
    print(row)


(0, 5022, 2790, 12.37, 2790, 12.41)
(1, 4824, 2680, 12.15, 2680, 12.19)
(2, 4905, 2666, 10.58, 2608, 10.55)
(3, 4932, 2188, 14.26, 2188, 14.23)
(4, 4950, 2199, 11.59, 2199, 11.53)
(5, 3944, 1971, 12.77, 1971, 12.81)
(6, 4544, 2271, 12.05, 2271, 12.03)


### Time-Based Operational Stress KPIs — `time_stress_kpis_month`, `time_stress_kpis_weekday`

**Business question**  
When are airline operations most stressed over time?

**How it works**
- Aggregates flights by time period (month, weekday)
- Reports flight volume and average delays
- Uses only valid delay records from the base view

**Key observations**
- **July** shows the highest average delays, indicating peak operational stress
- **August** has the highest volume but slightly lower delays, suggesting better capacity handling
- **September** has no usable delay data, as most flights were scheduled only
- Weekday delays are relatively stable, with a noticeable mid-week spike
- Departure and arrival delays closely track each other across time

**Usage**
- Identifying seasonal stress periods
- Staffing and capacity planning insights
- Line and bar charts in Tableau for trend analysis


In [10]:
# Volume vs delay relationship (daily)

cursor.executescript("""
DROP VIEW IF EXISTS volume_vs_delay_kpis;

CREATE VIEW volume_vs_delay_kpis AS
SELECT
    DATE(scheduled_departure_ts) AS flight_date,
    COUNT(*) AS total_flights,
    COUNT(dep_delay_min) AS flights_with_dep_delay,
    ROUND(AVG(dep_delay_min), 2) AS avg_dep_delay_min,
    COUNT(arr_delay_min) AS flights_with_arr_delay,
    ROUND(AVG(arr_delay_min), 2) AS avg_arr_delay_min
FROM analytics_flights_base
GROUP BY flight_date
ORDER BY flight_date;
""")

conn.commit()


In [11]:
cursor.execute("""
SELECT *
FROM volume_vs_delay_kpis
ORDER BY total_flights DESC
LIMIT 10;
""")

for row in cursor.fetchall():
    print(row)


('2017-07-22', 568, 568, 12.47, 568, 12.35)
('2017-07-29', 568, 568, 14.54, 568, 14.67)
('2017-08-05', 568, 568, 10.94, 568, 10.94)
('2017-08-12', 568, 567, 10.25, 567, 10.16)
('2017-08-19', 568, 0, None, 0, None)
('2017-08-26', 568, 0, None, 0, None)
('2017-09-02', 568, 0, None, 0, None)
('2017-09-09', 568, 0, None, 0, None)
('2017-07-16', 558, 558, 11.77, 558, 11.79)
('2017-07-23', 558, 558, 12.91, 558, 12.96)


### Volume vs Delay KPIs — `volume_vs_delay_kpis`

**Business question**  
Is higher flight volume associated with worse on-time performance?

**How it works**
- Aggregates flights at daily level
- Reports total flight volume and average delays
- Includes only valid delay records

**Key observations**
- Daily flight volume is highly consistent
- High volume does not consistently lead to higher delays
- Similar volumes can show very different delay outcomes
- Operational efficiency plays a larger role than congestion alone

**Usage**
- Volume vs delay scatter plots
- Identifying inefficient operating days
- Supporting data-driven capacity planning


In [12]:
# Short-haul vs long-haul reliability KPIs

cursor.executescript("""
DROP VIEW IF EXISTS haul_reliability_kpis;

CREATE VIEW haul_reliability_kpis AS
SELECT
    CASE
        WHEN aircraft_code IN ('CR2', 'SU9', 'CN1') THEN 'Short-haul'
        WHEN aircraft_code IN ('763', '733', '319', '320', '321') THEN 'Long-haul'
        ELSE 'Other'
    END AS haul_type,

    COUNT(*) AS total_flights,
    COUNT(dep_delay_min) AS flights_with_dep_delay,
    ROUND(AVG(dep_delay_min), 2) AS avg_dep_delay_min,
    COUNT(arr_delay_min) AS flights_with_arr_delay,
    ROUND(AVG(arr_delay_min), 2) AS avg_arr_delay_min

FROM analytics_flights_base
GROUP BY haul_type;
""")

conn.commit()


In [13]:
cursor.execute("SELECT * FROM haul_reliability_kpis;")
for row in cursor.fetchall():
    print(row)

('Long-haul', 5686, 2874, 11.82, 2864, 11.88)
('Other', 610, 308, 12.87, 307, 12.93)
('Short-haul', 26825, 13583, 12.26, 13536, 12.25)


In [2]:
conn.close()

### Assumptions & Definitions

- Flight distance is not available in the dataset.
- Aircraft type (`aircraft_code`) is used as a practical proxy for haul length.
- Classification logic:
  - **Short-haul**: regional and narrow-body aircraft
  - **Long-haul**: wide-body aircraft
- This proxy reflects operational planning differences rather than exact distance.

### Short-Haul vs Long-Haul Reliability — `haul_reliability_kpis`

**Business question**  
Do short-haul and long-haul flights differ in operational reliability?

**How it works**
- Uses aircraft type as a proxy for flight distance
- Groups flights into short-haul, long-haul, and other categories
- Compares average departure and arrival delays

**Key observations**
- Short-haul flights show slightly higher average delays
- Long-haul flights are marginally more reliable
- Flight distance alone does not explain delay behavior

**Usage**
- Reliability comparison by flight type
- Supporting fleet and scheduling decisions
- Segmentation in Tableau dashboards


### Excel Validation Export

The following step exports selected KPI views to a multi-sheet Excel workbook.
Excel is used only for validation and sanity checks before dashboarding.
No transformations are performed outside SQL.


In [16]:
from openpyxl import Workbook

wb = Workbook()
wb.remove(wb.active)  # remove default sheet

def export_view_to_sheet(view_name, sheet_name):
    cursor.execute(f"SELECT * FROM {view_name};")
    rows = cursor.fetchall()
    cols = [desc[0] for desc in cursor.description]

    ws = wb.create_sheet(title=sheet_name)
    ws.append(cols)
    for row in rows:
        ws.append(row)

export_view_to_sheet("airport_delay_kpis", "airport_delay_kpis")
export_view_to_sheet("time_stress_kpis_month", "time_stress_kpis_month")
export_view_to_sheet("volume_vs_delay_kpis", "volume_vs_delay_kpis")

wb.save("excel_validation.xlsx")

print("Excel validation file created.")


Excel validation file created.


### Tableau Data Exports

The following step exports finalized SQL KPI views to CSV files for use in Tableau.
These CSVs are direct outputs of the SQL views and contain no additional transformations.
Excel validation is performed separately and is not used as a Tableau data source.


In [18]:
import csv
import os

os.makedirs("tableau", exist_ok=True)

def export_view_to_csv(view_name):
    cursor.execute(f"SELECT * FROM {view_name};")
    rows = cursor.fetchall()
    cols = [desc[0] for desc in cursor.description]

    file_path = f"tableau/{view_name}.csv"
    with open(file_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(cols)
        writer.writerows(rows)

    print(f"Exported: {file_path}")

views = [
    "airport_delay_kpis",
    "time_stress_kpis_month",
    "time_stress_kpis_weekday",
    "volume_vs_delay_kpis",
    "haul_reliability_kpis"
]

for v in views:
    export_view_to_csv(v)

print("All Tableau CSVs created.")


Exported: tableau/airport_delay_kpis.csv
Exported: tableau/time_stress_kpis_month.csv
Exported: tableau/time_stress_kpis_weekday.csv
Exported: tableau/volume_vs_delay_kpis.csv
Exported: tableau/haul_reliability_kpis.csv
All Tableau CSVs created.
